# MNIST handwritten-digit classifier

A feed-forward neural network that classifies 28x28 grayscale images of handwritten
digits (0-9) from the classic **MNIST** dataset, built with TensorFlow / Keras.

This notebook follows a simple 5-step plan:

1. **Prepare the data** - load, normalize, and split into training / validation / test sets
2. **Outline the model** - choose the layers and activation functions
3. **Set the optimizer and loss**
4. **Train** the model
5. **Test** its accuracy on unseen data

> Built as a learning project while working through the deep-learning section of a
> data-science course. Each step includes a short note on *why*, not just *how*.

## 0. Imports and setup

We fix the random seeds so the run is reproducible - anyone who runs this notebook
should get the same results.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Reproducibility: fix the random seeds so results are repeatable
np.random.seed(42)
tf.random.set_seed(42)

print('TensorFlow version:', tf.__version__)

TensorFlow version: 2.21.0


## 1. Prepare the data (load, normalize, split)

MNIST ships with Keras. Each image is 28x28 pixels, each pixel an integer 0-255.

In [9]:
mnist_dataset, mnist_info = tfds.load(
    name='mnist',
    with_info=True,
    as_supervised=True,
    builder_kwargs={'file_format': 'tfrecord'},
)

## according to mnist documentation, the train is around 60k values and test is around 10k
mnist_train,mnist_test = mnist_dataset['train'], mnist_dataset['test']

num_validation_samples = 0.1 * mnist_info.splits['train'].num_examples
num_validation_samples = tf.cast(num_validation_samples, tf.int64)

num_test_samples = mnist_info.splits['test'].num_examples
num_test_samples = tf.cast(num_test_samples, tf.int64)

def scale(image,label):
    image = tf.cast(image, tf.float32)
    image /= 255. ## the . basically says we need it to be a float
    return image, label

scaled_train_and_validation_data = mnist_train.map(scale)
test_data = mnist_test.map(scale)

## shuffling the data, we need it as random as possible

BUFFER_SIZE = 10000
shuffled_train_and_validation_data = scaled_train_and_validation_data.shuffle(BUFFER_SIZE)

validation_data = shuffled_train_and_validation_data.take(num_validation_samples)
train_data = shuffled_train_and_validation_data.skip(num_validation_samples)

**Normalize the pixels.** They range 0-255, so we scale them to 0-1. This keeps every
input on a comparable scale so no large-magnitude value dominates training, which lets
gradient descent converge faster and more stably.

**Create a validation set.** Keras gives us train + test, but we want three sets. We
carve a 10% validation slice out of the training data. The model *learns* on train, and
we *watch* validation loss to catch overfitting - the validation set is never trained on.

## 2. Outline the model and choose activations

A simple feed-forward network:

- **Flatten** turns each 28x28 image into a 784-length vector.
- Two **Dense + ReLU** hidden layers learn features (ReLU is the standard hidden activation).
- A **Dense + softmax** output layer with 10 neurons (one per digit) gives a probability
  for each class.

## 3. Set the optimizer and loss

- **Adam** - momentum + adaptive per-weight step sizes; the practical default optimizer.
- **Cross-entropy loss** - the standard loss for classification. We use the `sparse_`
  version because our labels are integers (0-9), not one-hot vectors.

## 4. Make it learn

**Early stopping** watches the validation loss and halts when it stops improving,
restoring the best weights. This guards against overfitting (memorizing noise) and
saves us from guessing the exact number of epochs.

## 5. Test the accuracy

The honest final score, measured on the test set the model has never seen.

## What I learned

- The full supervised-learning workflow end to end: preprocess -> model -> optimize -> train -> evaluate.
- **Why** each choice is made: normalization for stable training, a held-out validation
  set to detect overfitting, ReLU vs softmax activations, cross-entropy for classification,
  Adam as the optimizer, and early stopping to halt at the right time.
- How to read training vs validation curves and a confusion matrix to judge a model.

*Next ideas: try more/wider layers, add dropout, or compare optimizers.*